In [72]:
import argparse
import sys
from pathlib import Path
from typing import List, Dict, Any, Tuple
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import json

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    print("ERROR: wandb not installed. Please install with: pip install wandb")
    sys.exit(1)


# =============================================================================
# Data Extraction
# =============================================================================

# Define metrics by type
METRIC_CONFIGS = {
    # MAE-based metrics
    "size": {"type": "mae", "category": "basic_property_reconstruction"},
    "avg_degree": {"type": "mae", "category": "basic_property_reconstruction"},
    "gini": {"type": "mae", "category": "basic_property_reconstruction"},
    "diameter": {"type": "mae", "category": "basic_property_reconstruction"},
    "homophily": {"type": "mae", "category": "community_related_property_reconstruction"},
    "community_presence": {"type": "mae", "category": "community_related_property_reconstruction"},
    "edge_prob_matrix": {"type": "mae", "category": "community_related_property_reconstruction"},
    # Accuracy-based metrics
    "community_detection": {"type": "accuracy", "category": "community_related_property_reconstruction"},
}

# For backwards compatibility and grouping
TASKS_DICT = {
    "basic_property_reconstruction": ["size", "avg_degree", "gini", "diameter"],
    "community_related_property_reconstruction": ["homophily", "community_presence", "edge_prob_matrix", "community_detection"]
}

def fetch_runs_from_wandb(project_path: str) -> List[Dict[str, Any]]:
    """
    Fetch all runs from a wandb project.
    
    Parameters
    ----------
    project_path : str
        Wandb project path in format "entity/project" or just "project".
    
    Returns
    -------
    List[Dict[str, Any]]
        List of run data dictionaries.
    """
    api = wandb.Api()
    
    print(f"\n{'=' * 80}")
    print(f"FETCHING RUNS FROM WANDB PROJECT: {project_path}")
    print(f"{'=' * 80}")
    
    # Fetch runs
    runs = api.runs(project_path, filters={"state": "finished"})
    
    run_data = []
    
    for run in runs:
        # Get config and summary
        config = dict(run.config)
        summary = dict(run.summary)
        
        # Extract relevant data
        data = {
            "run_id": run.id,
            "run_name": run.name,
            "config": config,
            "summary": summary,
        }
        
        run_data.append(data)
        print(f"  ✓ {run.id} ({run.name})")
    
    print(f"\n{'=' * 80}")
    print(f"FOUND {len(run_data)} FINISHED RUNS")
    print(f"{'=' * 80}\n")
    
    return run_data


def get_nested_value(d: dict, key_path: str, default=None):
    """Get value from nested dict using '/' separated path."""
    keys = key_path.split('/')
    value = d
    for key in keys:
        if isinstance(value, dict) and key in value:
            value = value[key]
        else:
            return default
    return value


def identify_variable_columns(df: pd.DataFrame, group_col: str) -> Dict[str, List[str]]:
    """
    For each unique value in group_col, identify which columns vary.
    
    Parameters
    ----------
    df : pd.DataFrame
        Dataframe with all runs.
    group_col : str
        Column to group by (e.g., 'data_name').
    
    Returns
    -------
    Dict[str, List[str]]
        Mapping from group value to list of variable column names.
    """
    variable_cols = {}
    
    for group_val in df[group_col].unique():
        group_df = df[df[group_col] == group_val]
        
        # Find columns that vary within this group
        varying = []
        for col in df.columns:
            if col in [group_col, 'run_id', 'run_name']:
                continue
            try:
                # Check if column has more than one unique value
                if group_df[col].nunique() > 1:
                    varying.append(col)
            except:
                # Skip columns that can't be compared (e.g., lists, dicts)
                pass
        
        variable_cols[group_val] = varying
    
    return variable_cols


# =============================================================================
# Data Processing
# =============================================================================

def check_metric_present(summary: dict, metric_name: str) -> bool:
    """
    Check if a specific metric is present in the summary.
    
    Parameters
    ----------
    summary : dict
        Run summary from wandb.
    metric_name : str
        Name of the metric (e.g., 'size', 'community_detection').
    
    Returns
    -------
    bool
        True if the metric is present.
    """
    metric_config = METRIC_CONFIGS.get(metric_name)
    if not metric_config:
        return False
    
    if metric_config['type'] == 'mae':
        return f'test/mae_{metric_name}' in summary
    elif metric_config['type'] == 'accuracy':
        # For community_detection, the key is 'test/accuracy_community_detection'
        return f'test/accuracy_{metric_name}' in summary
    
    return False


def process_runs(run_data: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Process run data into a structured dataframe.
    Each run produces one row per metric found in the summary.
    
    Parameters
    ----------
    run_data : List[Dict[str, Any]]
        Raw run data from wandb.
    
    Returns
    -------
    pd.DataFrame
        Processed dataframe with one row per metric per run.
    """
    records = []
    
    for run in run_data:
        config = run['config']
        summary = run['summary']
        
        # Extract basic info (shared across all rows from this run)
        base_info = {
            'run_id': run['run_id'],
            'run_name': run['run_name'],
            'pretraining_task': config['pretrain/model/model_name'],
            'n_train': get_nested_value(config, 'n_train'),
            'mode': get_nested_value(config, 'mode'),
            'readout_type': get_nested_value(config, 'readout_type'),
            'lr': get_nested_value(config, 'lr'),
            'patience': get_nested_value(config, 'patience'),
            'batch_size': get_nested_value(config, 'batch_size'),
            'classifier_dropout': get_nested_value(config, 'classifier_dropout'),
        }
        
        # Check each metric individually
        for metric_name, metric_config in METRIC_CONFIGS.items():
            if check_metric_present(summary, metric_name):
                record = dict(base_info)
                record['metric'] = metric_name
                record['metric_type'] = metric_config['type']
                record['task_type'] = metric_config['category']
                
                if metric_config['type'] == 'mae':
                    # Extract MAE-based metrics
                    record['train_value'] = summary.get(f'{metric_name}/train_mae')
                    record['val_value'] = summary.get(f'{metric_name}/val_mae')
                    record['test_value'] = summary.get(f'test/mae_{metric_name}')
                    record['test_baseline'] = summary.get(f'test/baseline_mae_{metric_name}')
                    record['test_improvement'] = summary.get(f'test/improvement_{metric_name}')
                elif metric_config['type'] == 'accuracy':
                    # Extract accuracy-based metrics
                    # For community_detection, keys are: train/val/test + accuracy_community_detection
                    record['train_value'] = summary.get(f'train/accuracy_{metric_name}')
                    record['val_value'] = summary.get(f'val/accuracy_{metric_name}')
                    record['test_value'] = summary.get(f'test/accuracy_{metric_name}')
                    record['test_baseline'] = None  # No baseline for accuracy
                    record['test_improvement'] = None  # No improvement metric
                
                records.append(record)
    
    df = pd.DataFrame(records)
    
    print(f"\nExtracted {len(records)} metric entries from {len(run_data)} runs:")
    if len(df) > 0:
        print(f"  Task types: {df['task_type'].unique().tolist()}")
        print(f"  Metric types: {df['metric_type'].unique().tolist()}")
        print(f"  Metrics: {df['metric'].unique().tolist()}")
        print(f"  Breakdown:")
        for task_type in df['task_type'].unique():
            task_df = df[df['task_type'] == task_type]
            metrics = task_df['metric'].unique()
            print(f"    {task_type}: {len(task_df)} entries ({', '.join(metrics)})")
        print(f"  Pretraining tasks: {df['pretraining_task'].unique().tolist()}")
        print(f"  Modes: {df['mode'].unique().tolist()}")
        print(f"  N_train values: {sorted(df['n_train'].unique().tolist())}")
        print(f"  Readout types: {df['readout_type'].unique().tolist()}")
    
    return df

def select_best_readout(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each unique configuration (excluding readout_type and metric value), 
    aggregate rows that differ only in readout_type or the actual metric value.
    
    Aggregation strategy:
    - For MAE metrics: Select best (lowest) val_value across readout_types
    - For accuracy metrics: Select best (highest) val_value across readout_types
    - For rows with same readout: Create list of metric values
    
    Returns a dataframe with one row per (config, metric) combination.
    """
    # Identify variable columns per pretraining_task
    variable_cols = identify_variable_columns(df, 'pretraining_task')

    print("\n" + "=" * 80)
    print("VARIABLE COLUMNS PER PRETRAINING_TASK")
    print("=" * 80)
    for pretraining_task, cols in variable_cols.items():
        print(f"\n{pretraining_task}:")
        print(f"  {cols}")

    # Group by all columns except readout_type and metric value columns
    group_cols = ['pretraining_task', 'n_train', 'mode', 'metric', 'metric_type', 'task_type']
    value_cols = ['train_value', 'val_value', 'test_value', 'test_baseline', 'test_improvement']
    
    for col in df.columns:
        if col not in group_cols + value_cols + ['readout_type', 'run_id', 'run_name']:
            if df[col].nunique() > 1:
                group_cols.append(col)

    print("\n" + "=" * 80)
    print("GROUPING COLUMNS (excluding readout_type and metric values):")
    print("=" * 80)
    print(f"  {group_cols}")

    best_records = []
    grouped = df.groupby(group_cols, dropna=False)

    print("\n" + "=" * 80)
    print("SELECTING BEST READOUT PER METRIC FOR EACH GROUP")
    print("=" * 80)

    for group_key, group_df in grouped:
        if len(group_df) == 0:
            continue
        
        base_record = {k: v for k, v in zip(group_cols, group_key)}
        metric_type = base_record['metric_type']
        
        # For community_detection (accuracy), readout_type doesn't matter
        # Just aggregate all values - use test_value if val_value is missing
        if base_record['metric'] == 'community_detection':
            # For community detection, we might not have val_value, so use test_value
            valid_rows = group_df[group_df['test_value'].notna()]
            
            if len(valid_rows) == 0:
                # No valid test values, skip
                continue
            
            record = dict(base_record)
            record['best_readout'] = None  # Not applicable
            
            # Use mean for train/val if available, otherwise None
            train_vals = valid_rows['train_value'].dropna()
            val_vals = valid_rows['val_value'].dropna()
            record['train_value'] = train_vals.mean() if len(train_vals) > 0 else None
            record['val_value'] = val_vals.mean() if len(val_vals) > 0 else None
            
            # Collect all test values as a list
            test_values = valid_rows['test_value'].dropna().tolist()
            record['test_value'] = test_values if len(test_values) > 1 else (test_values[0] if len(test_values) == 1 else None)
            record['test_baseline'] = None
            record['test_improvement'] = None
            record['run_ids'] = valid_rows['run_id'].tolist() if 'run_id' in valid_rows.columns else []
            
            best_records.append(record)
            continue
        
        # For other metrics, filter out rows with missing val_value
        valid_rows = group_df.dropna(subset=['val_value'])
        
        if len(valid_rows) == 0:
            # No valid rows, skip
            continue
        
        # For MAE and other accuracy metrics with val_value
        if False:  # This block is now handled separately above
            pass
        else:
            # For MAE metrics, select best readout based on val_value
            if metric_type == 'mae':
                # Lower is better
                best_idx = valid_rows['val_value'].idxmin()
            else:
                # Higher is better (accuracy)
                best_idx = valid_rows['val_value'].idxmax()
            
            best_row = valid_rows.loc[best_idx]
            
            # Check if there are multiple rows with the same readout_type
            same_readout = group_df[group_df['readout_type'] == best_row['readout_type']]
            
            record = dict(base_record)
            record['best_readout'] = best_row['readout_type']
            record['run_id'] = best_row['run_id']
            record['run_name'] = best_row['run_name']
            
            if len(same_readout) > 1:
                # Multiple runs with same config - collect values as list
                record['train_value'] = same_readout['train_value'].tolist()
                record['val_value'] = same_readout['val_value'].tolist()
                record['test_value'] = same_readout['test_value'].tolist()
                record['test_baseline'] = same_readout['test_baseline'].tolist() if 'test_baseline' in same_readout else None
                record['test_improvement'] = same_readout['test_improvement'].tolist() if 'test_improvement' in same_readout else None
            else:
                # Single run
                record['train_value'] = best_row['train_value']
                record['val_value'] = best_row['val_value']
                record['test_value'] = best_row['test_value']
                record['test_baseline'] = best_row.get('test_baseline')
                record['test_improvement'] = best_row.get('test_improvement')
            
            best_records.append(record)

    best_df = pd.DataFrame(best_records)
    print(f"\nReduced from {len(df)} metric entries to {len(best_df)} unique (config, metric) selections")
    
    # Print summary by task_type
    if len(best_df) > 0:
        print("\nBreakdown by task_type:")
        for task_type in best_df['task_type'].unique():
            count = len(best_df[best_df['task_type'] == task_type])
            metrics = best_df[best_df['task_type'] == task_type]['metric'].unique()
            print(f"  {task_type}: {count} selections ({', '.join(metrics)})")
        
        # Special debug for community_detection
        cd_count = len(best_df[best_df['metric'] == 'community_detection'])
        if cd_count > 0:
            print(f"\n  ✓ Found {cd_count} community_detection entries")
        else:
            # Debug: check if we have any in the input
            input_cd = len(df[df['metric'] == 'community_detection'])
            print(f"\n  ⚠ WARNING: No community_detection in output, but {input_cd} entries in input")
            if input_cd > 0:
                print(f"    Sample input rows:")
                sample = df[df['metric'] == 'community_detection'].head(3)
                for _, row in sample.iterrows():
                    print(f"      {row['pretraining_task']}, mode={row['mode']}, test_value={row.get('test_value')}, val_value={row.get('val_value')}")
    
    return best_df

# =============================================================================
# Publication Quality Plotting for Property Reconstructions
# =============================================================================

import matplotlib as mpl

def get_scalar_value(val):
    """Extract scalar from value (which might be a list or scalar)."""
    if isinstance(val, list):
        return np.mean(val) if len(val) > 0 else np.nan
    return val

def create_comparison_plots(
    df: pd.DataFrame,
    variable_cols: Dict[str, List[str]],
    output_path: str = "property_recon_comparison.png"
):
    """
    Create comparison plots for linear and finetune-linear modes.
    Uses publication-quality color, sizing, legend, and axis conventions.
    """
    for task_type in df['task_type'].unique():
        task_df = df[df['task_type'] == task_type]
        metrics = sorted(task_df['metric'].unique())
        
        # Special ordering for community_related: put community_detection at the end
        if task_type == 'community_related_property_reconstruction' and 'community_detection' in metrics:
            # Remove community_detection and add it at the end
            mae_metrics = [m for m in metrics if m != 'community_detection']
            metrics = mae_metrics + ['community_detection']
            print(f"  Reordered metrics for community_related: {metrics}")

        # Output file
        base_name = output_path.rsplit('.', 1)[0]
        ext = output_path.rsplit('.', 1)[1] if '.' in output_path else 'png'
        task_output_path = f"{base_name}_{task_type}.{ext}"

        print(f"\n{'=' * 80}")
        print(f"Creating publication-quality plot for {task_type} ({metrics})")
        print(f"{'=' * 80}")

        _create_pubquality_plot_for_task_type(task_df, metrics, variable_cols, task_output_path, task_type)

# --- Helper: nice color mapping, order, and families --- #

# Group pretraining tasks into 'generative' and 'contrastive'
_GENERATIVE = [
    "gps_dgmae", "gps_graphmaev2", "gps_s2gae"
]
_CONTRASTIVE = [
    "gps_dgi", "gps_graphcl"
]

_PRETRAIN_FAMILY_SHORT = {
    "generative": "Generative Pretraining",
    "contrastive": "Contrastive Pretraining"
}
_PRETRAIN_FAMILY_MEMBERS = {
    "generative": _GENERATIVE,
    "contrastive": _CONTRASTIVE,
}

# Define color palettes for publication quality
_GENERATIVE_COLORS = mpl.colormaps['Reds'](np.linspace(0.7, 0.95, len(_GENERATIVE)))
_CONTRASTIVE_COLORS = mpl.colormaps['Blues'](np.linspace(0.55, 0.93, len(_CONTRASTIVE)))
_PRETRAIN_COLOR_DICT = {k: c for k, c in zip(_GENERATIVE, _GENERATIVE_COLORS)}
_PRETRAIN_COLOR_DICT.update({k: c for k, c in zip(_CONTRASTIVE, _CONTRASTIVE_COLORS)})

# A consistent order across all subplots
_PRETRAIN_ORDER = _GENERATIVE + _CONTRASTIVE

_PRETTY_TASK_NAME = {
    "gps_dgmae": "DGMAE",
    "gps_graphmaev2": "GraphMAE v2",
    "gps_s2gae": "S2GAE",
    "gps_dgi": "DGI",
    "gps_graphcl": "GraphCL"
}
def get_pretty_name(task):
    return _PRETTY_TASK_NAME.get(task, str(task))

def _create_pubquality_plot_for_task_type(
    df: pd.DataFrame,
    metrics: List[str],
    variable_cols: Dict[str, List[str]],
    output_path: str,
    task_type: str
):
    """Create a publication-quality plot for one task type with color family legend, font, and configuration."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle

    # Publication quality: readable fonts, tight layout, little whitespace, bold titles.
    mpl.rcParams.update({
        "axes.labelsize": 14,
        "axes.titlesize": 16,
        "xtick.labelsize": 12,
        "ytick.labelsize": 11,
        "legend.fontsize": 13,
        "font.family": "DejaVu Serif",
        "axes.linewidth": 1.1,
        "xtick.direction": 'out',
        "ytick.direction": 'out',
        "axes.titleweight": 'bold',
        "figure.titlesize": 22
    })

    n_metrics = len(metrics)
    # 2 rows (modes), n_metrics columns, bigger size with less whitespace
    height = 2.6 + 3 * n_metrics
    
    # Check if we have community_detection (needs special spacing)
    has_community_detection = 'community_detection' in metrics
    
    if has_community_detection and len(metrics) > 1:
        # Use GridSpec for custom spacing
        from matplotlib.gridspec import GridSpec
        fig = plt.figure(figsize=(4.6 * n_metrics, 8.6))
        
        # Create GridSpec with extra space before last column
        # wspace list has n_metrics-1 entries (spaces between columns)
        gs = GridSpec(2, n_metrics, figure=fig, hspace=0.35, 
                     width_ratios=[1]*n_metrics,
                     wspace=0.21,  # Will adjust manually
                     left=0.08, right=0.78, top=0.89, bottom=0.15)
        
        axes = []
        for row in range(2):
            row_axes = []
            for col in range(n_metrics):
                ax = fig.add_subplot(gs[row, col])
                row_axes.append(ax)
            axes.append(row_axes)
        axes = np.array(axes)
    else:
        # Standard subplots
        fig, axes = plt.subplots(2, n_metrics, figsize=(4.6 * n_metrics, 8.6), squeeze=False)
        plt.subplots_adjust(hspace=0.35, wspace=0.21, left=0.08, right=0.78, top=0.89, bottom=0.15)

    def is_effectively_nan(val):
        if isinstance(val, (list, np.ndarray)):
            arr = np.array(val)
            if arr.size == 0:
                return True
            arr = arr.flatten()
            return np.all(pd.isna(arr))
        return pd.isna(val)

    # Ensure 'pretraining_task' is string
    df['pretraining_task'] = df['pretraining_task'].astype(str)
    unique_pretraining_tasks = [pt for pt in _PRETRAIN_ORDER if pt in df['pretraining_task'].unique()]
    # If new/forgotten task appears, append at end in gray
    for pt in df['pretraining_task'].unique():
        if pt not in unique_pretraining_tasks:
            unique_pretraining_tasks.append(pt)

    pretraining_task_colors = {k: _PRETRAIN_COLOR_DICT.get(k, 'gray') for k in unique_pretraining_tasks}

    def create_config_id(row, pretraining_task):
        var_cols = variable_cols.get(pretraining_task, [])
        parts = [str(pretraining_task), f"n{row['n_train']}"]
        for col in var_cols:
            if col not in ['mode', 'readout_type', 'task_type', 'metric', 'metric_type']:
                val = row.get(col)
                if not is_effectively_nan(val):
                    if isinstance(val, (list, np.ndarray)):
                        arr = np.array(val)
                        try:
                            display_val = np.mean(arr)
                        except Exception:
                            display_val = str(arr)
                    else:
                        display_val = val
                    parts.append(f"{col}={display_val}")
        return " | ".join(parts)
    df['config_id'] = df.apply(lambda row: create_config_id(row, row['pretraining_task']), axis=1)

    # Mapping task to color, legend, and label
    family_map = {}
    for fam, tasks in _PRETRAIN_FAMILY_MEMBERS.items():
        for t in tasks:
            family_map[t] = fam

    mode_rows = [
        ('linear', 'scratch', 0, 'Linear Probe'),
        ('finetune-linear', 'scratch', 1, 'Finetune (Linear)')
    ]
    # For each metric, collect bars, order by pretraining families
    for target_mode, scratch_mode, row_idx, row_title in mode_rows:
        for metric_idx, metric in enumerate(metrics):
            ax = axes[row_idx, metric_idx]
            
            # Get metric type (mae or accuracy)
            metric_config = METRIC_CONFIGS.get(metric, {})
            metric_type = metric_config.get('type', 'mae')
            is_accuracy = (metric_type == 'accuracy')
            
            # Only include configs matching desired pretraining tasks and metric/mode
            mode_df = df[
                (df['mode'] == target_mode) &
                (df['metric'] == metric) &
                (df['pretraining_task'].isin(unique_pretraining_tasks))
            ]
            if len(mode_df) == 0:
                ax.text(0.5, 0.5, f'No data for {target_mode}',
                    ha='center', va='center', transform=ax.transAxes,
                    fontsize=15
                )
                ax.set_title(f'{metric} — {row_title}')
                continue

            # For bar grouping: group bar positions by pretraining family, keep ordering fixed
            bars_value = []  # Will be improvement % for MAE or accuracy for accuracy metrics
            bars_label = []
            bars_color = []
            bars_ptask = []
            bars_configid = []
            bars_family = []

            # Order by PRETRAIN_ORDER (generative then contrastive)
            for pt in unique_pretraining_tasks:
                rows = mode_df[mode_df['pretraining_task'] == pt]
                for _, row in rows.iterrows():
                    if is_accuracy:
                        # For accuracy metrics, use test_value directly (convert to percentage)
                        val = get_scalar_value(row['test_value'])
                        if not is_effectively_nan(val):
                            val = val * 100  # Convert to percentage
                    else:
                        # For MAE metrics, use test_improvement
                        val = get_scalar_value(row['test_improvement'])
                        if not is_effectively_nan(val):
                            val = np.clip(val, -100, 100)
                    
                    if not is_effectively_nan(val):
                        bars_value.append(val)
                        bars_configid.append(row['config_id'])
                        bars_label.append(get_pretty_name(pt))
                        bars_color.append(pretraining_task_colors[pt])
                        bars_ptask.append(pt)
                        bars_family.append(family_map.get(pt, "other"))
            n_bars = len(bars_value)
            # Find bar group slices for each family
            family_indices = {}
            idx = 0
            for fam in _PRETRAIN_FAMILY_MEMBERS.keys():
                tasks = _PRETRAIN_FAMILY_MEMBERS[fam]
                n = sum([pt in tasks for pt in bars_ptask])
                if n > 0:
                    family_indices[fam] = (idx, idx+n)
                    idx += n

            # No sorting by value anymore, order is type-wise now

            # Bar plotting
            x_pos = np.arange(n_bars)
            # Bar width large, edges for clarity
            ax.bar(
                x_pos, bars_value, color=bars_color, alpha=0.85,
                edgecolor='black', linewidth=1.1,
                width=0.82
            )

            # Draw fine vertical lines between family groups (no labels beneath bars)
            divider_height = max(max(bars_value, default=1), 100)
            for fam, (i_start, i_end) in family_indices.items():
                if i_start > 0:
                    ax.axvline(i_start-0.5, color='gray', alpha=0.5, linestyle='--', lw=1.6, zorder=0)
            
            # Scratch mean line
            scratch_df = df[(df['mode'] == scratch_mode) & (df['metric'] == metric)]
            scratch_vals = []
            scratch_mean = None
            if len(scratch_df) > 0:
                for _, row in scratch_df.iterrows():
                    if is_accuracy:
                        # For accuracy, use test_value directly
                        val = get_scalar_value(row['test_value'])
                        if not is_effectively_nan(val):
                            scratch_vals.append(val * 100)  # Convert to percentage
                    else:
                        # For MAE, use test_improvement
                        val = get_scalar_value(row['test_improvement'])
                        if not is_effectively_nan(val):
                            scratch_vals.append(np.clip(val, -100, 100))
                if len(scratch_vals) > 0:
                    scratch_mean = np.mean(scratch_vals)
                    ax.axhline(
                        y=scratch_mean, color='black',
                        linestyle=(0, (7, 4)), linewidth=2, alpha=0.85
                    )
            # labels, formatting
            ax.set_xlabel('')
            # Y-axis label: for MAE metrics on leftmost, for accuracy on its column
            if metric_idx == 0:
                # Leftmost column - always MAE improvement label
                ax.set_ylabel('Test Improvement\nover baseline (%)')
            elif is_accuracy and metric_idx > 0:
                # For accuracy metrics (not in first column), add their own label
                ax.set_ylabel('Test Accuracy (%)')
            else:
                ax.set_ylabel('')
            
            # Title: metric name only on top row (bold), mode below (normal weight)
            if row_idx == 0:
                # Split title: metric name bold, mode normal
                metric_display = metric.replace("_", " ").capitalize()
                # Add "(node)" suffix for community_detection
                if metric == "community_detection":
                    metric_display += " (node)"
                
                ax.text(0.5, 1.15, metric_display, 
                       transform=ax.transAxes, ha='center', va='bottom',
                       fontweight='bold', fontsize=16)
                ax.text(0.5, 1.02, row_title,
                       transform=ax.transAxes, ha='center', va='bottom',
                       fontweight='normal', fontsize=14)
                ax.set_title('')  # Clear default title
            else:
                ax.set_title(f'{row_title}', pad=8, fontweight='normal', fontsize=14)
            # Bar x labels: config id (compact), but only indexes shown for cleanliness;
            # full details should be in supplementary table
            ax.set_xticks(x_pos)
            if row_idx == 0:
                # Remove x-axis labels from top row to reduce repetition
                ax.set_xticklabels([])
            else:
                # Only show x-axis labels on bottom row
                ax.set_xticklabels(
                    [get_pretty_name(ptask) for ptask in bars_ptask], rotation=40, ha='right', fontsize=14, va='top'
                )
                for label in ax.get_xticklabels():
                    label.set_fontweight('normal')
            # Set y-axis limits and styling based on metric type
            if is_accuracy:
                # For accuracy: dynamic range based on data, with 10% headroom
                all_vals = list(bars_value) if len(bars_value) > 0 else []
                if scratch_mean is not None:
                    all_vals.append(scratch_mean)
                
                if len(all_vals) > 0:
                    max_val = max(all_vals)
                    y_max = min(105, max_val + 10)  # Add 10% points, cap at 105
                else:
                    y_max = 105
                
                ax.set_ylim(0, y_max)
                # Dynamic tick spacing
                if y_max <= 60:
                    tick_spacing = 10
                else:
                    tick_spacing = 25
                ax.set_yticks(np.arange(0, y_max + 1, tick_spacing))
                
                # Gradient background for accuracy (adjust to y_max)
                if y_max > 75:
                    ax.axhspan(0, 50, facecolor='lightcoral', alpha=0.08, zorder=0)
                    ax.axhspan(50, 75, facecolor='lightyellow', alpha=0.08, zorder=0)
                    ax.axhspan(75, y_max, facecolor='palegreen', alpha=0.08, zorder=0)
                elif y_max > 50:
                    ax.axhspan(0, 50, facecolor='lightcoral', alpha=0.08, zorder=0)
                    ax.axhspan(50, y_max, facecolor='lightyellow', alpha=0.08, zorder=0)
                else:
                    ax.axhspan(0, y_max, facecolor='lightcoral', alpha=0.08, zorder=0)
            else:
                # For improvement: -110 to 110% range with zero line
                ax.axhline(y=0, color='grey', linestyle='-', linewidth=1.1, alpha=0.65)
                ax.set_ylim(-110, 110)
                ax.set_yticks(np.arange(-100, 125, 50))  # -100, -50, 0, 50, 100
                # Highlight background for super negative/positive
                ax.axhspan(-110, 0, facecolor='lightgray', alpha=0.08, zorder=0)
                ax.axhspan(0, 110, facecolor='palegreen', alpha=0.06, zorder=0)
            
            # Grid
            ax.grid(axis='y', alpha=0.20, zorder=0, linestyle=':', linewidth=1.0)

    # Add extra spacing before community_detection column if present
    if has_community_detection and len(metrics) > 1:
        # Shift columns 0 to n-2 slightly to the left, and column n-1 to the right
        extra_gap = 0.04  # Additional gap before community_detection
        for row_idx in range(2):
            for col_idx in range(n_metrics):
                ax = axes[row_idx, col_idx]
                pos = ax.get_position()
                if col_idx < n_metrics - 1:
                    # Shift left columns slightly left to create space
                    ax.set_position([pos.x0 - extra_gap * 0.5, pos.y0, pos.width, pos.height])
                else:
                    # Shift community_detection column to the right
                    ax.set_position([pos.x0 + extra_gap * 0.5, pos.y0, pos.width, pos.height])

    # Construct large, functional color legend for pretraining tasks, by family
    from matplotlib.lines import Line2D
    from matplotlib.text import Text
    legend_elements = []
    legend_labels = []
    for idx, fam in enumerate(_PRETRAIN_FAMILY_MEMBERS.keys()):
        # Add spacing between families (but not before the first one)
        if idx > 0:
            legend_elements.append(Rectangle((0, 0), 1, 1, fc='none', edgecolor='none', linewidth=0))
            legend_labels.append('')  # Empty line for spacing between families
        # Add family divider (not shown, just creates block)
        legend_elements.append(Rectangle((0, 0), 1, 1, fc='none', edgecolor='none', linewidth=0))
        legend_labels.append(f"{_PRETRAIN_FAMILY_SHORT[fam]}:")  # Normal font
        for t in _PRETRAIN_FAMILY_MEMBERS[fam]:
            if t in pretraining_task_colors:
                color = pretraining_task_colors[t]
                display = get_pretty_name(t)
                legend_elements.append(Rectangle((0, 0), 1, 1, fc=color, edgecolor='black'))
                legend_labels.append(f"  {display}")
    # Add "Other (gray)" block if needed
    for t in unique_pretraining_tasks:
        fam = family_map.get(t, "other")
        if fam not in _PRETRAIN_FAMILY_MEMBERS:
            legend_elements.append(Rectangle((0,0), 1,1, fc='gray', edgecolor='black'))
            legend_labels.append("Other: " + get_pretty_name(t))
    
    # Add separator and "From Scratch" baseline
    legend_elements.append(Rectangle((0, 0), 1, 1, fc='none', edgecolor='none', linewidth=0))
    legend_labels.append('')  # Empty line for spacing
    legend_elements.append(Line2D([0], [0], color='black', linestyle=(0, (7, 4)), linewidth=2, alpha=0.85))
    legend_labels.append('From Scratch')  # Normal font

    # Place big legend outside the plot right-hand side (closer to plots)
    from matplotlib.font_manager import FontProperties
    title_font = FontProperties(weight='bold', size=15)
    
    # Adjust legend position based on whether we have community_detection spacing
    legend_x = 0.85 if has_community_detection and len(metrics) > 1 else 0.79
    
    fig.legend(
        handles=legend_elements, labels=legend_labels,
        loc="center left", bbox_to_anchor=(legend_x, 0.53),
        borderaxespad=0.0, fontsize=14, frameon=False, ncol=1,
        handlelength=1.5, handletextpad=0.8, columnspacing=0.5,
        title="Pretraining Families", title_fontproperties=title_font
    )

    # Improve figure overall title and spacing
    fig.suptitle(f"{task_type.replace('_', ' ').title()}", fontweight='bold', fontsize=25, y=1.0)

    # Tighten layout, avoid excess space (leave room for legend on right)
    plt.tight_layout(rect=[0, 0.08, 0.78, 0.96])
    # Save
    plt.savefig(output_path, dpi=450, bbox_inches='tight')
    print(f"\n✓ PUB-QUALITY Plot saved to {output_path}")
    plt.close()


In [73]:
WANDB_PROJECT = "property_recon_multi"
OUTPUT = "property_recon_comparison.png"
SAVE_CSV = False
    
# Fetch runs
run_data = fetch_runs_from_wandb(WANDB_PROJECT)

if len(run_data) == 0:
    print("ERROR: No runs found in project")

# Process runs
df = process_runs(run_data)
    
# Select best readout for each configuration
best_df = select_best_readout(df)
variable_cols = identify_variable_columns(best_df, 'pretraining_task')

# Save to CSV if requested
if SAVE_CSV:
    csv_path = OUTPUT.replace('.png', '.csv')
    best_df.to_csv(csv_path, index=False)
    print(f"\n✓ Dataframe saved to {csv_path}")


FETCHING RUNS FROM WANDB PROJECT: property_recon_multi
  ✓ wiilab2f (bright-wave-1)
  ✓ ptzvinpc (still-pond-2)
  ✓ ft86wcrc (wild-disco-3)
  ✓ zgfc9w5j (sunny-dew-4)
  ✓ ixfv95ls (swift-microwave-5)
  ✓ 07ortdqq (rural-pyramid-6)
  ✓ cqupqcqv (laced-snowflake-7)
  ✓ glgrbtec (dashing-glade-8)
  ✓ 8di37jij (treasured-mountain-9)
  ✓ w5rqniqv (pleasant-firefly-10)
  ✓ 0sjv77ne (pious-star-11)
  ✓ 6itfv719 (stoic-leaf-12)
  ✓ 3254zcan (astral-night-13)
  ✓ ff0wtk3d (wild-wood-14)
  ✓ fgc6bu7a (effortless-brook-15)
  ✓ u77d3r46 (jumping-pond-16)
  ✓ pl53d2is (graceful-snow-17)
  ✓ qu6dan55 (rural-gorge-18)
  ✓ c4ugqw92 (crimson-waterfall-19)
  ✓ 4jhekzs7 (prime-snowflake-20)
  ✓ a744wen4 (graceful-frog-21)
  ✓ 8vcbtz79 (brisk-violet-22)
  ✓ beb5bz14 (colorful-energy-23)
  ✓ tk7vhtiz (fluent-silence-24)
  ✓ t1g91i4u (wild-water-25)
  ✓ 18umd3h2 (young-bird-26)
  ✓ 71mkeb6f (upbeat-paper-27)
  ✓ es4lkzlb (lemon-sponge-28)
  ✓ tzy4bjhb (olive-voice-29)
  ✓ 2nylqbc1 (peachy-oath-30)
  ✓ vk2j

In [74]:
# Create plots (separate for basic and community-related)
create_comparison_plots(best_df, variable_cols, OUTPUT)

  Reordered metrics for community_related: ['community_presence', 'edge_prob_matrix', 'homophily', 'community_detection']

Creating publication-quality plot for community_related_property_reconstruction (['community_presence', 'edge_prob_matrix', 'homophily', 'community_detection'])


C:\Users\louis\AppData\Local\Temp\ipykernel_28236\3399631632.py:557: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['pretraining_task'] = df['pretraining_task'].astype(str)
C:\Users\louis\AppData\Local\Temp\ipykernel_28236\3399631632.py:583: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['config_id'] = df.apply(lambda row: create_config_id(row, row['pretraining_task']), axis=1)
C:\Users\louis\AppData\Local\Temp\ipykernel_28236\3399631632.py:850: UserWarning: This figure includes Axes that are not compat


✓ PUB-QUALITY Plot saved to property_recon_comparison_community_related_property_reconstruction.png

Creating publication-quality plot for basic_property_reconstruction (['avg_degree', 'diameter', 'gini', 'size'])


C:\Users\louis\AppData\Local\Temp\ipykernel_28236\3399631632.py:557: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['pretraining_task'] = df['pretraining_task'].astype(str)
C:\Users\louis\AppData\Local\Temp\ipykernel_28236\3399631632.py:583: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['config_id'] = df.apply(lambda row: create_config_id(row, row['pretraining_task']), axis=1)



✓ PUB-QUALITY Plot saved to property_recon_comparison_basic_property_reconstruction.png
